In [ ]:
import logging
from collections import defaultdict
from copy import copy, deepcopy
import numpy as np

from qiskit.circuit.library.standard_gates import SwapGate
from qiskit.circuit.quantumregister import Qubit
from qiskit.transpiler.basepasses import TransformationPass
from qiskit.transpiler.exceptions import TranspilerError
from qiskit.transpiler.layout import Layout
from qiskit.dagcircuit import DAGOpNode
from qiskit.converters import circuit_to_dag, dag_to_circuit

In [ ]:
def run(self, dag):
    """Run the SabreSwap pass on `dag`.

    Args:
        dag (DAGCircuit): the directed acyclic graph to be mapped.
    Returns:
        DAGCircuit: A dag mapped to be compatible with the coupling_map.
    Raises:
        TranspilerError: if the coupling map or the layout are not
        compatible with the DAG
    """
    if len(dag.qregs) != 1 or dag.qregs.get("q", None) is None:
        raise TranspilerError("Sabre swap runs on physical circuits only.")

    if len(dag.qubits) > self.coupling_map.size():
        raise TranspilerError("More virtual qubits exist than physical.")

    self.dist_matrix = self.coupling_map.distance_matrix


    # Preserve input DAG's name, regs, wire_map, etc. but replace the graph.

    mapped_dag = dag._copy_circuit_metadata()

    canonical_register = dag.qregs["q"]

    # Start algorithm from the front layer and iterate until all gates done.
    num_search_steps = 0
    front_layer = dag.front_layer() #sl This front layer contains 1qbg
    self.applied_predecessors = defaultdict(int)
    for _, input_node in dag.input_map.items(): 
        for successor in self._successors(input_node, dag):
            self.applied_predecessors[successor] += 1

    while front_layer:
        
        execute_gate_list = []

        # Remove as many immediately applicable gates as possible
        for node in front_layer:
            front_layer.remove(node)
            if len(node.qargs) == 2:  
                v0, v1 = node.qargs
                if (v0, v1) in cur_graph.edges():
                    continue
                self._apply_gate(mapped_dag, node, current_layout, canonical_register)  
                
                for successor in self._successors(node, dag):
                    self.applied_predecessors[successor] += 1
                    if self._is_resolved(successor):  
                        front_layer.append(successor)

                        
        swap_node = DAGOpNode(op=SwapGate(), qargs=best_swap)
        self._apply_gate(mapped_dag, swap_node, current_layout, canonical_register)
        change += 1
        if printOK:
            print(dag_to_circuit(mapped_dag))

        gate_storage[p0], gate_storage[p1] = gate_storage[p1], gate_storage[p0]  # swap the 2 qubit storages to reflect the swap

        #sl---------------------
        num_swap += 1
        if num_swap > self.coupling_map.size():
            print('@@', num_search_steps, num_swap, qct_progress)
            raise Exception ('Fallback is necessary!')
        #sl---------------------

        current_layout.swap(*best_swap)

        #sl---------------------
        v0, v1 = best_swap
        p0, p1 = current_layout._v2p[v0], current_layout._v2p[v1]
        temp_progress =  max(qct_progress[p0], qct_progress[p1]) + 3
        qct_progress[p0] = temp_progress
        qct_progress[p1] = temp_progress
        #sl---------------------

        num_search_steps += 1
        if num_search_steps % DECAY_RESET_INTERVAL == 0:
            self._reset_qubits_decay()
        else:
            self.qubits_decay[best_swap[0]] += DECAY_RATE
            self.qubits_decay[best_swap[1]] += DECAY_RATE

        if QCTVerify:
            self.qct_sol[num_search_steps] = ((p0,p1),)

        # Diagnostics
        logger.debug("SWAP Selection...")
        logger.debug("extended_set: %s", [(n.name, n.qargs) for n in extended_set])
        logger.debug("swap scores: %s", sqgm_swap_scores)
        logger.debug("best swap: %s", best_swap)
        logger.debug("qubits decay: %s", self.qubits_decay)

    self.property_set["final_layout"] = current_layout
    # print(current_layout)

    if not self.fake_run:

        for i in phy_qubits:
            if len(gate_storage[i]) > 0: 
                for node in gate_storage[i]:
                    self._apply_gate(mapped_dag, node, current_layout, canonical_register)  # apply the remaining gates
                    change += 1
                    if printOK:
                        print(dag_to_circuit(mapped_dag))


        # for key in self.qct_sol:
        #     print(key,self.qct_sol[key])

        if QCTVerify:
            self.verify(mapped_dag)
        return mapped_dag
    return dag


def _apply_gate(self, mapped_dag, node, current_layout, canonical_register):
    if self.fake_run:
        return
    new_node = _transform_gate_for_layout(node, current_layout, canonical_register)
    mapped_dag.apply_operation_back(new_node.op, new_node.qargs, new_node.cargs)

def _reset_qubits_decay(self):
    """Reset all qubit decay factors to 1 upon request (to forget about
    past penalizations).
    """
    self.qubits_decay = {k: 1 for k in self.qubits_decay.keys()}

def _successors(self, node, dag):
    for _, successor, edge_data in dag.edges(node):
        if not isinstance(successor, DAGOpNode):
            continue
        if isinstance(edge_data, Qubit):
            yield successor

def _is_resolved(self, node):
    """Return True if all of a node's predecessors in dag are applied."""
    return self.applied_predecessors[node] == len(node.qargs)

def _obtain_extended_set(self, dag, front_layer):
    #sl front_layer only contains 2-qubit gates
    """Populate extended_set by looking ahead a fixed number of gates.
    For each existing element add a successor until reaching limit.
    """
    extended_set = []
    incremented = []
    tmp_front_layer = front_layer
    done = False
    while tmp_front_layer and not done:
        new_tmp_front_layer = []
        for node in tmp_front_layer:
            for successor in self._successors(node, dag):
                incremented.append(successor)
                self.applied_predecessors[successor] += 1 #sl temporally 
                if self._is_resolved(successor):
                    new_tmp_front_layer.append(successor)
                    if len(successor.qargs) == 2:
                        extended_set.append(successor)
            if len(extended_set) >= self.EXTENDED_SET_SIZE: 
                done = True
                break
        tmp_front_layer = new_tmp_front_layer
    for node in incremented:
        self.applied_predecessors[node] -= 1 #sl set back
    return extended_set

def _obtain_swaps(self, front_layer, current_layout):
    """Return a set of candidate swaps that affect qubits in front_layer.

    For each virtual qubit in front_layer, find its current location
    on hardware and the physical qubits in that neighborhood. Every SWAP
    on virtual qubits that corresponds to one of those physical couplings
    is a candidate SWAP.

    Candidate swaps are sorted so SWAP(i,j) and SWAP(j,i) are not duplicated.
    """
    candidate_swaps = set()
    for node in front_layer:
        for virtual in node.qargs: 
            physical = current_layout[virtual] 
            for neighbor in self.coupling_map.neighbors(physical):
                virtual_neighbor = current_layout[neighbor] 
                swap = sorted([virtual, virtual_neighbor], key=lambda q: self._bit_indices[q])
                candidate_swaps.add(tuple(swap))

    return candidate_swaps

def _compute_cost(self, layer, layout):
    cost = 0
    layout_map = layout._v2p
    for node in layer:
        if len(node.qargs) == 1: 
            raise Exception ('The layer contains no 1-qubit gates!', node.qargs, len(node.qargs))

        cost += self.dist_matrix[layout_map[node.qargs[0]], layout_map[node.qargs[1]]]
    return cost


def verify(self, mapped_dag):
    out_circ = dag_to_circuit(mapped_dag)
    # print('wo',type(out_circ))

    # Extract the gate list
    gate_list = []
    for instruction in out_circ:
        if instruction[0].name == 'measure':
            continue  # Skip measurement instructions
        if len(instruction[1]) == 1:
            gate_list.append((instruction[1][0].index,))
        elif len(instruction[1]) == 2:
            gate_list.append((instruction[1][0].index, instruction[1][1].index))

    # print(gate_list)

    # circ_qasm = CreateCircuitFromQASM(filename, path)
    # circ = ReducedCircuit(circ_qasm) #1&2qbg list


    QC = qctCircuit(AG, gate_list)
    qubit_num_circ = QC.num_qubit 
    cx_gates = QC.cxcirc
    # print(cx_gates)
    QC.qct_verify(self.qct_sol)

def _transform_gate_for_layout(op_node, layout, device_qreg):
    """Return node implementing a virtual op on given layout."""
    mapped_op_node = copy(op_node)

    premap_qargs = op_node.qargs
    mapped_qargs = map(lambda x: device_qreg[layout._v2p[x]], premap_qargs)
    mapped_op_node.qargs = list(mapped_qargs)

return mapped_op_node

